In [1]:
import torch
import torch.nn as nn

import torch_geometric as tg
import torch_geometric.utils as tgu
import torch_geometric.nn as tgnn
import matplotlib.pyplot as plt

import numpy as np

In [2]:
omega = 1.8

feature_vec_len = 8
x = np.arange(0,10,0.1).reshape((-1,1))
y = np.arange(0,10,0.1).reshape((-1,1))
t = np.arange(0,6,0.1).reshape([-1,1,1])

xy = np.sqrt(x**2 + y.T**2).reshape([1,x.shape[0],y.shape[0]])

values = np.sin(omega*t + xy)

edge_embedding_size = 4 
edge_params = torch.rand([*xy.shape,4])
bias = torch.rand([*xy.shape, feature_vec_len])

In [3]:
a = torch.zeros(1000,1000)

In [60]:
base_samples = [(10,10), (90,10), (10,90), (90,90)]
# values refers to measured value -> shape [time, *xy.shape]
# xy refers to class of measurment
# t refers to time of measurment
def get_sample(values, xy, t):
    number_of_additional_nodes = np.random.randint(2,6)
    number_of_nodes_to_predict = np.random.randint(2,5)
    prob_base = np.random.rand()
    rand_coordinates = set(base_samples if prob_base>0.3 else [] + [tuple(np.random.randint(0,values.shape[1], [2,])) for _ in range(number_of_additional_nodes)])
    cord_to_predict = []
    it = 0
    while number_of_nodes_to_predict > len(cord_to_predict):
        it += 1
        candidate = tuple(np.random.randint(0,values.shape[1], [2,]))
        if candidate not in rand_coordinates:
            cord_to_predict.append(candidate)
        if it>20:
            raise Exception("something is wrong wit while loop")

    time = np.random.randint(0,50)
    dt = np.random.randint(4, 60-time-1)

    inp_known = []
    inp_unknown = []
    out_t_0 = []
    out_t_dt = []
    list_of_node_classes = []

    for cord in rand_coordinates:
        inp_known.append((*cord, t, values[t,*cord]))
        out_t_0.append((*cord, t, values[t,*cord]))
        out_t_dt.append((*cord, t+dt, values[t+dt,*cord]))
        list_of_node_classes.append(cord)


    for cord in cord_to_predict:
        inp_unknown.append(cord)
        out_t_0.append((*cord, t, values[t,*cord]))
        out_t_dt.append((*cord, t+dt, values[t+dt,*cord]))
        list_of_node_classes.append(cord)

    return (
        (tgu.dense_to_sparse(torch.ones(len(out_t_0), len(out_t_0)))[0], list_of_node_classes),
        (torch.tensor(inp_known, dtype=torch.float32), torch.tensor(inp_unknown, dtype=torch.int32)),
        torch.tensor(out_t_0, dtype=torch.float32),
        torch.tensor(out_t_dt, dtype=torch.float32),
        torch.tensor([[time], [time+dt]]).to(torch.float32),
    )

In [61]:
class feature_encoder(nn.Module):
    def __init__(self, dims=None, output_feature_num=feature_vec_len):
        nn.Module.__init__(self)

        if dims is None:
            self.dims = {
                "time": 1,
                "x_pos": 1,
                "y_pos": 1,
                "value": 1,
            } 
        else:
            self.dims = dims

        self.input_dimensionality = sum(self.dims.values())

        self.seq = nn.Sequential(
            nn.Linear(self.input_dimensionality, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
            nn.Linear(32, output_feature_num),
        )

    def forward(self, x):
        return self.seq(x)

In [62]:
fe_model = feature_encoder()
te = feature_encoder({'time': 1}, output_feature_num=16)
sample = get_sample(values, xy, 10)
fe_model(sample[1][0])

tensor([[-9.9956e-01,  4.9500e-01,  1.8366e+00,  1.5185e+00, -1.4028e+00,
          1.2494e+00,  9.3509e-01,  6.1668e-01],
        [-3.4594e+00, -2.5665e+00,  5.6285e+00,  7.7494e+00, -2.5366e+00,
          6.0851e+00,  3.9139e+00,  4.7382e+00],
        [-7.5823e-02,  1.3883e-03,  8.9807e+00,  2.7495e+00, -2.2231e+00,
          9.0957e+00, -2.4948e-01,  2.5062e+00],
        [-4.8170e+00, -1.4408e+00,  8.9922e+00,  1.0444e+01, -5.2078e+00,
          8.1826e+00,  2.9349e+00,  4.1943e+00]], grad_fn=<AddmmBackward0>)

In [70]:
torch.rand(*np.array([2,2]))

tensor([[0.1975, 0.5954],
        [0.9136, 0.3094]])

In [73]:
# x_cord, y_cord, time
# time_embed 


class gnn_model(nn.Module):
    def __init__(self):
        nn.Module.__init__(self)
        self.edge_vector_len = 4
        self.gnn_out_ch = 16
        self.edge_params = nn.Parameter(torch.rand([*(xy.shape[1:]*2), self.edge_vector_len]))
        self.bias = nn.Parameter(torch.rand([*xy.shape[1:], feature_vec_len]))
        self.fe = feature_encoder()
        self.te = feature_encoder({'time': 1}, output_feature_num=self.gnn_out_ch)
        self.l1 = tgnn.GAT(
            in_channels=feature_vec_len,
            hidden_channels=4*feature_vec_len,
            num_layers=4,
            out_channels=self.gnn_out_ch,
            dropout=0.1,
            act='gelu'
        )

    @staticmethod
    def map_edges_to_class_cords(edges, mapping):
        cords = []
        for c1, c2 in edges.T:
            cords.append([mapping[c1], mapping[c2]])
        
        tensor_cords = torch.tensor(cords).flatten(start_dim=1)
        cords_as_separate_rows = tensor_cords.T
        return cords_as_separate_rows

    def forward(self, inp_known, inp_unknown, t, edge_index, edge_index_cls_mapping):
        edges_class_cords = self.map_edges_to_class_cords(edge_index, edge_index_cls_mapping)
        edge_params_for_graph = self.edge_params[*edges_class_cords, :]
        encoded_features = self.fe(inp_known)
        bias_features = self.bias [*inp_unknown.T,:]
        inp = torch.cat([encoded_features, bias_features])
        g_output = self.l1(inp, edge_index,  edge_attr=edge_params_for_graph)
        t_embed = self.te(t)
        return g_output@(t_embed.T)


In [74]:
model = gnn_model()

In [75]:
model(
    inp_known=sample[1][0],
    inp_unknown=sample[1][1],
    t=sample[-1],
    edge_index=sample[0][0],
    edge_index_cls_mapping=sample[0][1]
)

tensor([[0.2974, 0.4180],
        [0.4143, 0.5820],
        [0.4624, 0.6473],
        [0.3926, 0.5575],
        [0.4081, 0.5735],
        [0.1710, 0.2483],
        [0.4898, 0.6859]], grad_fn=<MmBackward0>)

In [ ]:
NUM_TR_STEPS = 1024
loss_t_0 = []
loss_t_dt = []

tensor([[34],
        [41]])